In [2]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
import pandas as pd
import numpy as np
import joblib
import os

In [3]:
# Evaluation metrics used in the paper:

def paper_r2(y_true, y_pred):
    """Paper's metric: Pearson correlation"""
    corr = np.corrcoef(y_true, y_pred)[0, 1]
    return corr 

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred)**2))

In [4]:
def fit_csv (csv_filename='individual_proteins_dataset', selected_attributes=None, categorical_attributes=None, protein_targets=None, show=False):

    """
    Resumen de funcionalidad:
    1. Carga el dataset y preprocesa los datos: filtra los atributos seleccionados y aplica One-Hot Encoding a las variables categóricas.
    2. Gestiona el sistema de archivos verificando la existencia del directorio './modelos_guardados/', creándolo si es necesario.
    3. Itera sobre cada proteína objetivo (target) especificada.
    4. Entrena un modelo RandomForestRegressor definitivo para cada objetivo, utilizando la totalidad de los datos disponibles (sin división de validación).
    5. Serializa y guarda cada modelo entrenado en formato .joblib en el directorio especificado.
    6. Retorna la estructura de columnas (X_train_columns) para asegurar que los datos futuros tengan el mismo formato antes de predecir.
    """
    
    if selected_attributes == None or categorical_attributes == None or protein_targets == None:
        print("No se han especificado los atributos seleccionados, los Atributos Categóricos o los Atributos Objetivos")
        return;

    # ======================================== Inicialización ============================================= #
    
    # Load dataset: 
    df = pd.read_csv(csv_filename + '.csv')
    
    # Get selected attributes from dataset:
    X = df[selected_attributes]
    
    # One-hot encode categorical attributes:
    X = pd.get_dummies(X, columns=categorical_attributes)
    
    # Se guardan los nombres de las columnas
    X_train_columns = X.columns
    
    # Directorio en donde se van a guardar los modelos
    directory = './modelos_guardados/'
    
    # Si el directorio no existe, lo crea
    if not os.path.exists(directory):
        os.makedirs(directory)
        print(f"Directorio creado: {directory}")

    # ===================================================================================================== #

    for target in protein_targets:
        
        # Valores de las Proteinas/Targets
        y = df[target].values
    
        # Inicialización y Ajuste del Modelo RF
        model = RandomForestRegressor(n_estimators=500, random_state=42, max_samples=0.3) 
        model.fit(X, y)

        # Nombre del modelo a Guardar
        filename = f'random_forest_regressor_model_{target}.joblib'
    
        filepath = os.path.join(directory, filename)
        joblib.dump(model, filepath)

        # Muestra cada Modelo y la dirección en donde se guarda dicho Modelo
        if show:
            print(f"Modelo: {filename}")
            print(f"Se guardo exitosamente en: {filepath}")
            
    print(f"\n========================================================================")
    print(f"Se han guardado con exito todos los modelos en: {directory}")

    return X_train_columns

In [5]:
def predict_csv(csv_filename='datos_np_unq', selected_attributes=None, categorical_attributes=None, protein_targets=None, categorical_columns=None , show=False):

    """
    Resumen de funcionalidad:
    1. Carga el dataset de nuevos datos y aplica el preprocesamiento inicial: selección de variables y One-Hot Encoding.
    2. Alinea forzosamente la estructura de columnas del dataset de prueba con la del entrenamiento (`categorical_columns`), rellenando con 0 las categorías faltantes para asegurar la compatibilidad exacta con los modelos guardados.
    3. Itera sobre cada proteína objetivo, cargando desde disco (`./modelos_guardados/`) el modelo Random Forest entrenado correspondiente.
    4. Genera las predicciones para cada proteína y las incorpora como nuevas columnas al dataset original (con el sufijo '_prediccion').
    5. Guarda el dataset enriquecido (datos originales + predicciones) en un nuevo archivo CSV de salida.
    """
    
    if selected_attributes == None or categorical_attributes == None or protein_targets == None:
        print("No se han especificado los atributos seleccionados, los Atributos Categóricos o los Atributos Objetivos")
        return;

    
    # ======================================== Inicialización ============================================= #
    
    # Load dataset: 
    dj = pd.read_csv(csv_filename + '.csv')
    
    # Get selected attributes from dataset:
    X_test = dj[selected_attributes]
    
    # One-hot encode categorical attributes:
    X_test = pd.get_dummies(X_test, columns=categorical_attributes)

    # Se añaden los atributos categoricos faltantes
    X_test = X_test.reindex(columns=categorical_columns, fill_value=0)

    # Directorio de donde se van a cargar los modelos
    directory = './modelos_guardados/'
    
    # ===================================================================================================== #
    
    for target in protein_targets:

        # Lectura del Modelo
        filename = f'random_forest_regressor_model_{target}.joblib'
        
        filepath = os.path.join(directory, filename)
        loaded_model = joblib.load(filepath)

        # Predicción del Modelo Cargado con datos de prueba
        loaded_prediction = loaded_model.predict(X_test)

        # Concatenación del resultado de la predicción de la Proteina con el csv original
        dj[target + '_prediccion'] = loaded_prediction

        # Muestra la Predicción de la Proteina.
        if show:
            print(f"============= Proteina: {target} =============")
            print(f"Predicción de la Proteina {target}: {loaded_prediction}")

    # Al finalizar, guarda el csv con el resultado de las predicciones.
    output_filename = csv_filename + '_con_predicciones.csv'
    dj.to_csv(output_filename, index=False)
    
    print("\n--- PROCESO TERMINADO ---")
    print(f"El archivo final se guardó como: {output_filename}")
    print(f"Columnas añadidas: {len(protein_targets)}")

In [6]:
# Definition of attributes and targets

selected_attributes = [
    'np_without_modification', 'surface_modification', 'zeta_potential', 
    'incubation_protein_source', 'incubation_plasma_concentration', 'incubation_np_concentration',
    'np_type', 'np_shape', 'dispersion_medium', 'dispersion_medium_ph', 'size_dls', 'pdi', 
    'incubation_culture', 'incubation_time', 'incubation_temperature', 'modification_type'
]

categorical_attributes = [
    'np_without_modification', 'surface_modification', 'incubation_protein_source',
    'np_type', 'np_shape', 'dispersion_medium', 'incubation_culture', 'modification_type'
]

protein_targets = [
    "P01871", "P01024", "P02647", "P02649", "P02768", "P04004", "P01834", 
    "P10909", "P02652", "P00734", "P01009", "P01042", "P01857", "P01859", 
    "P01023", "P01619", "P04196", "P02656", "P04114", "P02787", "P02766", 
    "P06396", "P06727", "P0C0L5", "P02671", "P08603", "P02765", "Q14624", 
    "P0DOY2", "P01008", "P68871", "P01764", "P04003", "P0C0L4", "P01876", 
    "P02749", "P02675", "P01860", "P01011", "P00747", "P02774", "P00751", 
    "P19823", "P08697", "P19827", "P69905", "P02760", "P02748", "P02679", 
    "P02790", "B9A064", "P07996", "P05155", "P01766", "P12259", "P00738",
    "P02751", "P02654", "P04406", "P00739", "P02655", "P00736", "P07225", 
    "P05154", "P09871", "P60709", "P03952", "P02747", "P05546", "P02746",
    "P27169", "P01019", "P35542", "P04217", "Q14520", "P02743", "P02763", 
    "P49908", "O43866", "P18428", "P55056", "P04264", "P00748", "P01591", 
    "P01861", "Q92954", "P02776", "P0DJI8", "P01031", "P05156", "P13671", 
    "Q03591", "P06312", "P00740", "P01615", "P00742", "P04070", "O14791", 
    "P05090", "P02745", "P00488", "Q13103", "P01877", "P01700", "P10643", 
    "Q9UK55", "P03951", "Q06033", "Q96IY4", "P13645", "P04433", "P07358",
    "P27918", "P05452", "P20851", "P07357", "P07360", "P01599", "O95445", 
    "Q9BXR6", "P02775", "P02741", "P35579", "P15169", "Q13790", "P35858", 
    "P49747", "P36955", "P19652", "P07737", "P22891", "P35908", "P80748", 
    "P08514", "P01593", "Q04756", "P06733", "P23528", "P63104", "P18065", 
    "P08519", "Q86UX7", "P02753", "Q5TB80", "P22352", "P00746", "P35443", 
    "P62937", "P10720", "P48740", "P25311", "P43652", "P35527", "Q9Y490", 
    "P05106", "P17936", "P18206", "P02788", "P06702", "P01701", "Q6Q788", 
    "Q96PD5", "Q9UGM5", "O00391", "P11142", "P14618", "P23142", "P68366",
    "P12814", "P61224", "Q13201", "P67936", "P0DOY3", "P08709", "P01034", 
    "P81605", "P02533", "P11226"]

In [8]:
columns = fit_csv(selected_attributes=selected_attributes, categorical_attributes=categorical_attributes, protein_targets=protein_targets, show=True)

Modelo: random_forest_regressor_model_P01871.joblib
Se guardo exitosamente en: ./modelos_guardados/random_forest_regressor_model_P01871.joblib
Modelo: random_forest_regressor_model_P01024.joblib
Se guardo exitosamente en: ./modelos_guardados/random_forest_regressor_model_P01024.joblib
Modelo: random_forest_regressor_model_P02647.joblib
Se guardo exitosamente en: ./modelos_guardados/random_forest_regressor_model_P02647.joblib
Modelo: random_forest_regressor_model_P02649.joblib
Se guardo exitosamente en: ./modelos_guardados/random_forest_regressor_model_P02649.joblib
Modelo: random_forest_regressor_model_P02768.joblib
Se guardo exitosamente en: ./modelos_guardados/random_forest_regressor_model_P02768.joblib
Modelo: random_forest_regressor_model_P04004.joblib
Se guardo exitosamente en: ./modelos_guardados/random_forest_regressor_model_P04004.joblib
Modelo: random_forest_regressor_model_P01834.joblib
Se guardo exitosamente en: ./modelos_guardados/random_forest_regressor_model_P01834.joblib

In [9]:
predict_csv(csv_filename="datos_np_unq_controles" ,selected_attributes=selected_attributes, categorical_attributes=categorical_attributes, protein_targets=protein_targets, categorical_columns=columns, show=True)

============= Proteina: P01871 =============
Predicción de la Proteina P01871: [0.9159055  0.86442106 0.90466666 0.91511732 2.20579129 1.94836783
 2.19946541]
============= Proteina: P01024 =============
Predicción de la Proteina P01024: [1.65758727 1.6990226  1.73666442 1.77220419 0.42867223 0.99774818
 1.2134164 ]
============= Proteina: P02647 =============
Predicción de la Proteina P02647: [2.02859283 2.08626102 2.0083042  1.99818336 1.4882637  1.03921054
 0.92454184]
============= Proteina: P02649 =============
Predicción de la Proteina P02649: [1.15912324 1.15784284 1.16899718 1.17446587 4.13601446 3.7197287
 1.08817283]
============= Proteina: P02768 =============
Predicción de la Proteina P02768: [4.63207031 4.55318533 4.34301463 4.32177388 2.23098861 2.0271526
 5.12840288]
============= Proteina: P04004 =============
Predicción de la Proteina P04004: [1.23375198 1.34320107 1.38590376 1.39812013 0.05894645 0.17140694
 1.57130292]
============= Proteina: P01834 =============
Pre

C:\Users\emanu\AppData\Local\Temp\ipykernel_11216\3750159842.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dj[target + '_prediccion'] = loaded_prediction


============= Proteina: O14791 =============
Predicción de la Proteina O14791: [0.01489009 0.0183575  0.02898173 0.02794125 0.         0.00029407
 0.0839458 ]


C:\Users\emanu\AppData\Local\Temp\ipykernel_11216\3750159842.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dj[target + '_prediccion'] = loaded_prediction


============= Proteina: P05090 =============
Predicción de la Proteina P05090: [0.17338988 0.08101319 0.06513943 0.07541062 0.00546    0.00054
 0.03549907]


...

C:\Users\emanu\AppData\Local\Temp\ipykernel_11216\3750159842.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dj[target + '_prediccion'] = loaded_prediction
C:\Users\emanu\AppData\Local\Temp\ipykernel_11216\3750159842.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dj[target + '_prediccion'] = loaded_prediction


============= Proteina: P08709 =============
Predicción de la Proteina P08709: [0.         0.0003262  0.0003262  0.0003262  0.00010795 0.
 0.00532543]
============= Proteina: P01034 =============
Predicción de la Proteina P01034: [0.0004914 0.        0.        0.        0.        0.0138572 0.       ]


C:\Users\emanu\AppData\Local\Temp\ipykernel_11216\3750159842.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dj[target + '_prediccion'] = loaded_prediction
C:\Users\emanu\AppData\Local\Temp\ipykernel_11216\3750159842.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dj[target + '_prediccion'] = loaded_prediction


============= Proteina: P81605 =============
Predicción de la Proteina P81605: [0.00305168 0.00305168 0.00146903 0.00097348 0.         0.0081751
 0.04079407]
============= Proteina: P02533 =============
Predicción de la Proteina P02533: [0.08978991 0.04059993 0.01201631 0.01201631 0.002      0.00045197
 0.19302923]
============= Proteina: P11226 =============
Predicción de la Proteina P11226: [0.         0.         0.         0.         0.18768    0.02930397
 0.04510959]

--- PROCESO TERMINADO ---
El archivo final se guardó como: datos_np_unq_controles_con_predicciones.csv
Columnas añadidas: 178


C:\Users\emanu\AppData\Local\Temp\ipykernel_11216\3750159842.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dj[target + '_prediccion'] = loaded_prediction
